# Fluora on Colab (GPU)

Runs the fluora single-cell fluorescence pipeline on a Colab GPU to compare against the ~157s/frame CPU baseline.

**Before running:** `Runtime` → `Change runtime type` → **T4 GPU** (T4 is sm_75, which Colab's PyTorch supports). Then `Runtime` → `Run all`.

The install cell runs **first, before anything imports numpy**, so the numpy/scipy upgrade pulled in by cellpose is in place before use. If you ever hit a `numpy ... _blas_supports_fpe` error, it means numpy got upgraded after import — just `Runtime` → `Restart session` and `Run all` again.

In [ ]:
# 1. Install deps FIRST, before any numpy import (Colab has numpy/pandas/torch).
#    -U scipy keeps scipy consistent with the numpy cellpose pulls in.
!pip install -q -U cellpose laptrack tifffile scipy

In [ ]:
# 2. Confirm a usable GPU (need compute capability sm_75+)
import torch
assert torch.cuda.is_available(), 'No GPU — set Runtime > Change runtime type > T4 GPU'
cap = torch.cuda.get_device_capability(0)
print('GPU:', torch.cuda.get_device_name(0), '| sm_%d%d' % cap, '| torch', torch.__version__)
assert cap >= (7, 5), f'GPU sm_{cap[0]}{cap[1]} too old for this torch build (needs sm_75+)'
print('OK — GPU is supported')

In [ ]:
# 3. Clone the repo and enter it (we run the package in-place; no install,
#    which sidesteps the project's Python>=3.14 pin vs Colab's Python)
![ -d fluora ] || git clone https://github.com/phuongho43/fluora.git
%cd fluora

## Input data
Cell 4 generates the same synthetic frames used to validate the pipeline locally (numpy-only, no scipy), so this runs end-to-end with no upload.

**To use your own timelapse instead:** skip cell 4, and upload your `.tiff` files into an `imgs/` folder with each filename being its timestamp in seconds (e.g. `0.0.tiff`, `30.0.tiff`, ...).

In [ ]:
# 4. Generate synthetic fluorescence frames (4 drifting cells, distinct intensities)
from pathlib import Path
import numpy as np, tifffile

def blur(img, sigma=2.0):  # numpy-only separable Gaussian (avoids scipy)
    r = int(3 * sigma)
    k = np.exp(-(np.arange(-r, r + 1) ** 2) / (2 * sigma ** 2)); k /= k.sum()
    for ax in (0, 1):
        img = np.apply_along_axis(lambda v: np.convolve(v, k, mode='same'), ax, img)
    return img

OUT = Path('imgs'); OUT.mkdir(exist_ok=True)
H, W = 256, 256
RNG = np.random.default_rng(0)
cells = [
    {'pos': np.array([60.0, 60.0]),  'vel': np.array([3.0, 2.0]),   'r': 14, 'ampl': 180},
    {'pos': np.array([180.0, 70.0]), 'vel': np.array([-2.0, 3.0]),  'r': 16, 'ampl': 120},
    {'pos': np.array([90.0, 190.0]), 'vel': np.array([2.5, -2.0]),  'r': 12, 'ampl': 220},
    {'pos': np.array([190.0, 185.0]),'vel': np.array([-3.0, -2.5]), 'r': 15, 'ampl': 90},
]
yy, xx = np.mgrid[0:H, 0:W]
for f in range(5):
    img = np.zeros((H, W))
    for c in cells:
        cy, cx = c['pos'] + c['vel'] * f
        img += c['ampl'] * ((yy - cy) ** 2 + (xx - cx) ** 2 <= c['r'] ** 2)
    img = blur(img, sigma=2.0) + RNG.normal(20, 5, (H, W))
    tifffile.imwrite(OUT / f'{f * 30.0}.tiff', np.clip(img, 0, None).astype(np.uint16))
print('wrote', len(list(OUT.glob('*.tiff'))), 'frames to imgs/')

In [ ]:
# 5. Run the actual fluora pipeline on GPU, timing each frame
import time
from fluora.io import load_images
from fluora.track import track_cells
from fluora.extract import extract_intensities
from cellpose import models

timestamps, images = load_images('imgs')
model = models.CellposeModel(gpu=True)  # device='cuda' path from segment.py

masks = []
for i, img in enumerate(images):
    t = time.time()
    mask = model.eval(img)[0]
    torch.cuda.synchronize()
    masks.append(mask.astype('int32'))
    print(f'  frame {i+1}/{len(images)}: {time.time()-t:5.2f}s  ({int(mask.max())} cells)')

track_df = track_cells(masks)
traces = extract_intensities(timestamps, images, masks, track_df)
traces.to_csv('traces.csv', index=False)
print(f'\n{traces.cell_id.nunique()} cell traces, {len(traces)} rows')
traces

Compare the per-frame times above (expect ~1–3s on a T4) against the local CPU baseline of ~157s/frame. The `traces` table should show 4 cells, each with a stable distinct intensity across all 5 timepoints.